In [1]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
os.makedirs("dataset_outputs/A2AR/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="A2AR/data/a2ar_train_1",
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [3]:
display(dataset.X.shape)
display(dataset.X_ind.shape)
display(dataset.getDF())
display(dataset.X)


(2436, 0)

(0, 0)

,QSPRID,Y,Drug,Y_original
QSPRID,,,,
A2ARDataset_0000,A2ARDataset_0000,True,Cc1cc(C)n(-c2cc(NC(=O)CCN(C)C)nc(-c3ccc(C)o3)n...,True
A2ARDataset_0001,A2ARDataset_0001,True,CNC(=O)C12CC1C(n1cnc3c(NCc4cccc(Cl)c4)nc(C#CCC...,True
A2ARDataset_0002,A2ARDataset_0002,True,CCNC(=O)C1OC(n2cnc3c(NCC)nc(C#CCCCc4ccccc4)nc3...,True
A2ARDataset_0003,A2ARDataset_0003,True,Cc1cc(C)n(-c2cc(NC(=O)CN3CCOCC3)nc(-c3ccc(C)o3...,True
A2ARDataset_0004,A2ARDataset_0004,True,COc1ccc(N2CCN(CCn3c(=O)n(C)c4c3nc(N)n3nc(-c5cc...,True
...,...,...,...,...
A2ARDataset_2431,A2ARDataset_2431,True,CNc1ncc(C(=O)NCc2ccc(OC)cc2)c2nc(-c3ccco3)nn12,True
A2ARDataset_2432,A2ARDataset_2432,True,Nc1nc(-c2ccco2)c2ncn(C(=O)NCCc3ccccc3)c2n1,True
A2ARDataset_2433,A2ARDataset_2433,False,Nc1nc(CSc2nnc(N)s2)nc(Nc2ccc(F)cc2)n1,False


""
QSPRID
A2ARDataset_0000
A2ARDataset_0001
A2ARDataset_0002
A2ARDataset_0003
A2ARDataset_0004
...
A2ARDataset_2431
A2ARDataset_2432
A2ARDataset_2433


In [4]:
import torch
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset

# Nastavení zařízení (GPU nebo CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Načtení tokenizeru a modelu
tokenizer = RobertaTokenizerFast.from_pretrained("entropy/roberta_zinc_480m", max_len=128)
model = RobertaForMaskedLM.from_pretrained('entropy/roberta_zinc_480m')

# Přenesení modelu na správné zařízení
model.to(device)

# Připravení collatoru pro padding
collator = DataCollatorWithPadding(tokenizer, padding=True, return_tensors='pt')

# Načtení seznamu SMILES (například z nějaké jiné struktury než datasetu)
smiles = dataset.getDF()["Drug"]

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding

smiles_dataset = SMILESDataset(smiles, tokenizer)

# Vytvoření DataLoaderu (dávky po 32)
batch_size = 32
dataloader = DataLoader(smiles_dataset, batch_size=batch_size, collate_fn=collator)

# Zpracování dat po dávkách
model.eval()  # Převede model do evaluačního režimu (bez trénování)
embeddings_list = []  # Uchováme všechny embeddings

with torch.no_grad():  # Nevytvářet gradienty během evaluace
    for batch in dataloader:
        # Zkontroluj tvar batchů
        print("Batch input_ids tvar:", batch['input_ids'].shape)
        print("Batch attention_mask tvar:", batch['attention_mask'].shape)

        # Přenesení všech vstupů na správné zařízení (GPU nebo CPU)
        input_ids = batch['input_ids'].squeeze(1).to(device)  # Squeeze odstraní extra dimenzi
        attention_mask = batch['attention_mask'].squeeze(1).to(device)  # Squeeze odstraní extra dimenzi

        # Modelování výstupů s hidden states
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        # Získání poslední vrstvy hidden states
        full_embeddings = outputs[1][-1]

        # Výpočet průměrného embeddingu pro každý token
        embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
        
        # Uložení embeddings pro tuto dávku
        embeddings_list.append(embeddings)

# Spojení všech embeddings z dávky do jednoho tensoru
all_embeddings = torch.cat(embeddings_list, dim=0)

# Teď můžeš použít `all_embeddings`, což obsahuje embeddings pro všechny SMILES v datasetu


/tmp/ipykernel_10674/2991385132.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch inpu

In [5]:
import pandas as pd

# Převod na NumPy pole a pak na DataFrame
df = pd.DataFrame(all_embeddings.cpu().numpy())


In [6]:
dataset.X = pd.concat([dataset.X, df], axis=1)


/tmp/ipykernel_10674/1146750015.py:1: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  dataset.X = pd.concat([dataset.X, df], axis=1)


In [7]:
display(dataset.X.shape)

(4872, 768)

In [8]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [9]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        # Zjistíme embedding dimenzi na prvním SMILES
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("A2AR/data/a2ar_train_1")

X2_all = load_datasets("A2AR/data/a2ar_val_1")

X3_all = load_datasets("A2AR/data/a2ar_test")

transformer = ChemBERTaTransformer()
X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
X_val_emb = transformer.transform(X2_all.df["Drug"])
X_test_emb = transformer.transform(X3_all.df["Drug"])


/tmp/ipykernel_10674/195630644.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


In [11]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
X_train_emb = X_train_emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, X_train_emb], axis = 1)

In [12]:
X2_all.X = X2_all.X.reset_index(drop=True)
X_val_emb = X_val_emb.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, X_val_emb], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
X_test_emb = X_test_emb.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, X_test_emb], axis = 1)

In [13]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [14]:
imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [15]:
pd.DataFrame(X1).columns[pd.DataFrame(X1).isna().any()].tolist()



[]

In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001
0,-0.099751,-0.317440,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,-0.683718,-0.154789,-0.263361,...,-0.282544,0.422059,-1.822235,-0.056128,1.794965,-1.565456,-1.048544,0.309060,0.151831,-1.744244
1,-0.099751,-0.317440,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,-0.683718,-0.154789,-0.263361,...,-0.843830,-0.196947,1.520924,0.514490,-0.037110,0.885964,1.188376,0.079622,0.330666,-1.601492
2,-0.099751,-0.317440,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,-0.683718,-0.154789,-0.263361,...,-1.291498,0.231219,0.188872,1.016670,-0.870611,-0.990171,-0.993077,1.673972,-0.847206,-0.075383
3,-0.099751,-0.317440,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,-0.683718,-0.154789,-0.263361,...,-0.072665,0.694371,-1.242190,-0.044025,0.985496,-1.711103,-1.385791,0.311699,-0.228046,-1.850311
4,-0.099751,-0.317440,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,1.462592,6.460406,-0.263361,...,-1.032917,0.504506,0.206682,0.477171,-0.223751,1.323563,-1.297888,0.440150,-0.207804,-1.023053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3514,-0.099751,2.681493,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,1.462592,-0.154789,-0.263361,...,0.012980,-0.312655,0.226109,0.456055,-0.095439,0.315570,1.298719,-0.326961,1.355913,1.507767
3515,-0.099751,-0.317440,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,-0.683718,-0.154789,-0.263361,...,0.505704,-0.132930,2.605090,0.300434,-1.294903,-1.726489,0.960560,-1.940551,0.959784,-1.269714
3516,-0.099751,-0.317440,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,-0.683718,-0.154789,-0.263361,...,-0.257429,-0.439072,0.636566,-0.361596,0.162856,-1.288887,0.213336,-0.306234,-1.465708,1.233436
3517,-0.099751,3.150201,-0.111664,-0.125883,-0.095465,-0.035115,-0.07036,-0.683718,-0.154789,-0.263361,...,-0.916666,-0.532430,1.034437,-0.989490,-0.054451,0.494912,-0.900077,1.267733,-0.551359,-1.747319


In [17]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [18]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, matthews_corrcoef
import pandas as pd

def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    val_mcc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        val_mcc_t.append(matthews_corrcoef(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
        print(matthews_corrcoef(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    my_df["MCC"] = val_mcc_t
    return my_df

In [33]:
import optuna
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
import torch
import torch.nn.functional as F
import torch.optim as optim

def objective(trial, X_train, y_train, X_test, y_test):
    # Parametry z původního gridu
    #act_fun = trial.suggest_categorical("act_fun", [F.selu])
    dropout_frac = trial.suggest_categorical("dropout_frac", [0,0.1, 0.2, 0.4, 0.5, 0.6, 0.8, 0.9])
    patience = trial.suggest_categorical("patience", [10, 40, 75])
    tol = trial.suggest_categorical("tol", [1e-5, 1e-4, 1e-3, 1e-2, 0])
    weight_decay = trial.suggest_categorical("weight_decay", [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 0])
    n_epochs = trial.suggest_categorical("n_epochs", [200, 300, 500, 1000])
    neuron_layers = trial.suggest_categorical("neuron_layers", [
    [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],  # původní
    [2048, 1024, 512, 256, 128, 64, 32, 16, 8],        # menší
    [4096, 1024, 256, 64, 8],                          # rychlejší zúžení
    [4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8],  # hladké
    [4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096],
        [200], [2000], [2000, 1000], [2000, 1000, 500],[1000, 50], [4000, 2000],[4000, 2000, 1000, 500],
        [4000, 2000, 2000, 500]
    ])
    ch_size = trial.suggest_categorical("batch_size", [1024, 512, 256, 128, 64])
    optimizer = trial.suggest_categorical("optimizer", [optim.AdamW, optim.RMSprop])
    lr = trial.suggest_categorical("lr", [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    # Model
    model = STFullyConnected(
        n_dim=X_train.shape[1],
        n_class=1,
        gpus=[],
        device=device,
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=dropout_frac,
        patience=patience,
        tol=1e-5,
        weight_decay=weight_decay,
        n_epochs=n_epochs,
        neuron_layers=neuron_layers,
        batch_size=batch_size,
        optimizer=optimizer,
        lr=lr
    )

    # Trénink a predikce
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_bin = preds > 0.5

    # Metiky
    f1 = f1_score(y_test, preds_bin)
    acc = accuracy_score(y_test, preds_bin)
    mcc = matthews_corrcoef(y_test, preds_bin)

    # Můžeš logovat i do trialu
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("acc", acc)

    return mcc  # maximalizujeme MCC


In [ ]:
study_3 = optuna.create_study(direction="maximize")
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=25
)

print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=50
)

print("Best MCC:", study.best_value)
print("Best parameters:", study.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study.best_trial.user_attrs["f1"])
print("Best ACC:", study.best_trial.user_attrs["acc"])


[I 2025-04-22 20:58:03,295] A new study created in memory with name: no-name-5af7cff5-64a7-4af7-b163-051d272a0d16
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <function selu at 0x7f9cb19e0360> which is of type function.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent s

cuda


[I 2025-04-22 21:02:55,126] Trial 0 finished with value: 0.4005694153876801 and parameters: {'act_fun': <function selu at 0x7f9cb19e0360>, 'dropout_frac': 0.1, 'patience': 40, 'tol': 1e-05, 'weight_decay': 0.0001, 'n_epochs': 300, 'neuron_layers': [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 256, 'optimizer': <class 'torch.optim.rmsprop.RMSprop'>, 'lr': 1e-05}. Best is trial 0 with value: 0.4005694153876801.
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <function selu at 0x7f9cb19e0360> which is of type function.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [

cuda


[I 2025-04-22 21:09:26,711] Trial 1 finished with value: 0.024638455838810993 and parameters: {'act_fun': <function selu at 0x7f9cb19e0360>, 'dropout_frac': 0.1, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001, 'n_epochs': 300, 'neuron_layers': [4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 256, 'optimizer': <class 'torch.optim.rmsprop.RMSprop'>, 'lr': 0.001}. Best is trial 0 with value: 0.4005694153876801.
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <function selu at 0x7f9cb19e0360> which is of type function.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent sto

...

cuda


[I 2025-04-22 23:35:28,286] Trial 40 finished with value: 0.40931463246325 and parameters: {'act_fun': <function selu at 0x7f9cb19e0360>, 'dropout_frac': 0.1, 'patience': 40, 'tol': 1e-05, 'weight_decay': 0.0001, 'n_epochs': 300, 'neuron_layers': [2048, 1024, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 128, 'optimizer': <class 'torch.optim.rmsprop.RMSprop'>, 'lr': 0.0001}. Best is trial 4 with value: 0.48074454428754815.
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <function selu at 0x7f9cb19e0360> which is of type function.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [4096,

cuda


[I 2025-04-22 23:38:27,755] Trial 41 finished with value: 0.48074454428754815 and parameters: {'act_fun': <function selu at 0x7f9cb19e0360>, 'dropout_frac': 0.4, 'patience': 40, 'tol': 1e-05, 'weight_decay': 0.0001, 'n_epochs': 300, 'neuron_layers': [2048, 1024, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 128, 'optimizer': <class 'torch.optim.rmsprop.RMSprop'>, 'lr': 0.0001}. Best is trial 4 with value: 0.48074454428754815.
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <function selu at 0x7f9cb19e0360> which is of type function.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [40

cuda


In [35]:
study_2 = optuna.create_study(direction="maximize")
study_2.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=25
)

print("Best MCC:", study_2.best_value)
print("Best parameters:", study_2.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_2.best_trial.user_attrs["f1"])
print("Best ACC:", study_2.best_trial.user_attrs["acc"])


[I 2025-04-24 00:43:43,726] A new study created in memory with name: no-name-a1d1b6e7-479c-4f39-a9f6-992ba9cdb0ca
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persist

cuda


[I 2025-04-24 00:49:51,809] Trial 0 finished with value: 0.0 and parameters: {'dropout_frac': 0.4, 'patience': 75, 'weight_decay': 0.0001, 'n_epochs': 500, 'neuron_layers': [4096, 1024, 256, 64, 8], 'batch_size': 128, 'lr': 0.5}. Best is trial 0 with value: 0.0.
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-pack

cuda


[I 2025-04-24 00:53:02,917] Trial 1 finished with value: 0.0 and parameters: {'dropout_frac': 0.4, 'patience': 40, 'weight_decay': 1e-05, 'n_epochs': 300, 'neuron_layers': [2048, 1024, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 256, 'lr': 0.1}. Best is trial 0 with value: 0.0.
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/pyt

...

cuda


[I 2025-04-24 02:20:22,145] Trial 23 finished with value: 0.45750101555936595 and parameters: {'dropout_frac': 0.55, 'patience': 40, 'weight_decay': 1e-05, 'n_epochs': 300, 'neuron_layers': [2048, 1024, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 256, 'lr': 0.0001}. Best is trial 4 with value: 0.4895147622708506.
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [2048, 1024, 512, 256, 128, 64, 32, 16, 8] which is of type list.
  warnings.warn(message)
/home/ubuntu/m

cuda


[I 2025-04-24 02:25:04,642] Trial 24 finished with value: 0.32740773828608644 and parameters: {'dropout_frac': 0.55, 'patience': 40, 'weight_decay': 1e-05, 'n_epochs': 300, 'neuron_layers': [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 256, 'lr': 0.0001}. Best is trial 4 with value: 0.4895147622708506.


Best MCC: 0.4895147622708506
Best parameters: {'dropout_frac': 0.4, 'patience': 40, 'weight_decay': 0.0001, 'n_epochs': 300, 'neuron_layers': [2048, 1024, 512, 256, 128, 64, 32, 16, 8], 'batch_size': 256, 'lr': 0.0001}
Best F1: 0.9809586342744583
Best ACC: 0.9633375474083439
